# ArabicDeXlit — Train Your Own Arabic ASR De-Transliterator

> **Undo Arabic ASR transliteration:** turn `انترن` back into `intern`, `ايه اي` back into `AI` —
> while leaving genuinely Arabic text **byte-identical**.

Arabic ASR models transcribe spoken English using Arabic letters. This notebook trains a small
two-stage model that repairs that, as a drop-in post-processor.

| | |
|---|---|
| **Input** | `كنت انترن في او اي اي و بعدها جالي اوفر` |
| **Output** | `كنت intern في OIE و بعدها جالي offer` |
| **Pure Arabic in** | `أنا رايح البيت دلوقتي` |
| **Pure Arabic out** | `أنا رايح البيت دلوقتي` *(unchanged, guaranteed)* |

### How it works

One **end-to-end ByT5** model. The noisy sentence goes in, the corrected sentence comes out,
and every decision is made by cross-attention -- no detector, no router, no rules.

Byte-level means the vocabulary is 256 symbols, so **nothing is ever out of vocabulary**:
`C++` is three bytes, `الsystem` needs no segmentation, and a brand name never seen in
training is still representable exactly.

| | |
|---|---|
| `عندي ميتينج مع المانجر بكرة` | `عندي Meeting مع ال Manager بكرة` |
| `انك تكتب كود ب سي بلاس بلاس` | `انك تكتب Code ب C++` |
| `نجهز البريزنتيشن` | `نجهز ال Presentation` |

Pure Arabic is learned as an identity mapping and checked explicitly by the
**Unnecessary Modification Rate**, the metric that decides whether the model is safe to deploy.

### What you need

- **Runtime → Change runtime type → GPU.** T4, L4 or A100 all work.
- A [Weights & Biases](https://wandb.ai) account (optional, for monitoring).
- Nothing else — the dataset is downloaded ready-built from the Hub.

| GPU | Model | Approx. time |
|---|---|---|
| A100 40GB | `google/byt5-base` (580M) | ~4–6 h for 3 epochs |
| L4 / T4 | `google/byt5-small` (300M) | set `--model-name google/byt5-small` |

Byte-level sequences are long, so this is slower than the old pipeline -- that is the cost of
letting one model see the whole sentence.

## 1. Check the GPU

In [ ]:
!nvidia-smi

import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)[0]
    print("GPU:", torch.cuda.get_device_name(0))
    # Ampere and newer (A100, L4) have real bf16; T4 must use fp16. The trainer
    # picks this automatically -- shown here so you know what you're getting.
    print("autocast dtype:", "bfloat16" if cap >= 8 else "float16")
else:
    print("No GPU detected. Runtime -> Change runtime type -> GPU")

## 2. Get the code

In [ ]:
import os

# Re-anchor first. Deleting the repo while the kernel is inside it leaves the
# process with a working directory that no longer exists, and every later
# command then fails with "getcwd: cannot access parent directories" -- even
# git clone, which cannot resolve where to create the folder.
for base in ("/teamspace/studios/this_studio", "/content", os.path.expanduser("~")):
    if os.path.isdir(base):
        os.chdir(base)
        break
print("working from:", os.getcwd())

REPO = "Arabic-DeXlit"
if not os.path.isdir(REPO):
    !git clone -q https://github.com/MohammedAly22/Arabic-DeXlit.git
else:
    !cd $REPO && git pull -q        # already cloned: take the latest
%cd $REPO
!pip install -q -r requirements.txt

print()
print("ready:", os.getcwd())

## 3. Load the dataset from the Hugging Face Hub

The corpus is already built and published, so there is nothing to generate here — this pulls
~68 MB of Parquet and writes the JSONL splits the training scripts read.

**[`mohammedaly22/ArabicDeXlit-Corpus`]( https://huggingface.co/datasets/mohammedaly22/ArabicDeXlit-Corpus)**

| | |
|---|---|
| Examples | 475,945 |
| Dialects | 8 (MSA, Egyptian, Gulf, Levantine, Iraqi, Maghrebi, Sudanese, Yemeni) |
| Pass-through rows | 30% — pure Arabic, where the correct answer is *change nothing* |
| Leakage | none: 0 overlapping sentence families between train / validation / test |

In [ ]:
import sys
sys.path.insert(0, "src")
from arabic_dexlit.data.hub import download_corpus, corpus_summary

download_corpus("data/processed", repo_id="mohammedaly22/ArabicDeXlit-Corpus")

import json
print(json.dumps(corpus_summary("data/processed"), indent=2))

In [ ]:
# A look at what the model actually learns from.
import json

rows = [json.loads(l) for l in open("data/processed/validation.jsonl", encoding="utf-8").readlines()[:400]]

conv = next(r for r in rows if r["spans"])
print("--- a conversion example ---")
print("INPUT :", conv["src"])
print("TARGET:", conv["tgt"])
print("TAGS  :", list(zip(conv["src_tokens"], conv["tags"])))

keep = next(r for r in rows if all(t == "O" for t in r["tags"]))
print("\n--- a pass-through example (must not change) ---")
print("INPUT :", keep["src"])
print("TARGET:", keep["tgt"])
print("identical:", keep["src"] == keep["tgt"])

<details>
<summary><b>Optional: rebuild the dataset from scratch instead</b></summary>

You only need this if you want to change how the data is made — a different pass-through ratio,
more augmentation variants, or extra dialect synthesis. It downloads the source corpora and
regenerates everything (10–20 minutes).

```python
!python scripts/build_dataset.py --fetch-mono 150000 --passthrough-ratio 0.30 --variants 1
```

To add more dialect and acronym/email coverage with Gemini
(free key from [Google AI Studio](https://aistudio.google.com/apikey)):

```python
import os
os.environ["GEMINI_API_KEY"] = "..."
!python scripts/synthesize_data.py --total 30000 --workers 16
!python scripts/build_dataset.py --passthrough-ratio 0.30
```
</details>

## 4. Weights & Biases (optional)

Tracks loss, accuracy, the safety metrics and the diagnostic plots. Skip this cell to train without logging.

In [ ]:
WANDB_PROJECT = "arabic-dexlit"   # set to None to disable

if WANDB_PROJECT:
    import wandb
    wandb.login()   # paste your key from https://wandb.ai/authorize

## 5. Smoke test

Runs the whole pipeline on a few hundred examples in about a minute.

**Always run this before starting a long GPU session.** It catches a broken path immediately
rather than forty minutes in.

In [ ]:
# Tiny run on 64 examples -- proves the whole path works before a long session.
cmd = (
    "python scripts/train.py --stage seq2seq --config configs/seq2seq_t4.yaml "
    "--model-name google/byt5-small --max-train-examples 64 --max-eval-examples 8 "
    "--batch-size 2 --eval-batch-size 2 --epochs 1 --eval-every 10 --log-every 5 "
    "--output-dir outputs/smoke --no-wandb"
)
print(cmd)
!{cmd}

### Optional: measure your GPU first

Binary-searches the largest batch that actually fits, times gradient checkpointing on and off,
and checks whether the dataloader is starving the GPU. Prints a recommended config.

In [ ]:
!python scripts/diagnose_gpu.py --model google/byt5-base

## 6. Train the end-to-end rewriter

One model, one objective. Watch **UMR** (Unnecessary Modification Rate) above everything else:
it is the fraction of tokens that should have been left alone but were changed. For a drop-in ASR
post-processor a missed conversion is recoverable; a corrupted word is not.

Checkpoints are selected on `sentence_exact - UMR`, so a model that improves conversion while
damaging untouched Arabic cannot win.

In [ ]:
import torch

# Per-GPU configs. The generic default left an H200 at 13GB of 150GB and
# 0.78 steps/s; these set batch size, checkpointing and worker count for the
# card actually present.
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
name = torch.cuda.get_device_name(0)
if vram > 70:
    CONFIG = "configs/seq2seq_h100.yaml"     # H100 / H200 / A100 80GB
elif vram > 30:
    CONFIG = "configs/seq2seq_a100.yaml"     # A100 40GB
else:
    CONFIG = "configs/seq2seq_t4.yaml"       # T4 / L4 -- byt5-small
print(f"{name} ({vram:.0f}GB) -> {CONFIG}")

wandb_arg = f"--wandb-project {WANDB_PROJECT}" if WANDB_PROJECT else "--no-wandb"
cmd = (
    f"python scripts/train.py --stage seq2seq --config {CONFIG} "
    f"--output-dir outputs/dexlit-s2s {wandb_arg}"
)
print(cmd)
!{cmd}

## 7. Try it

The first block is the real test: pure Arabic must come back **character for character identical**.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from arabic_dexlit.model.seq2seq import Seq2SeqConfig, generate

cfg = Seq2SeqConfig()
tok = AutoTokenizer.from_pretrained("outputs/dexlit-s2s/seq2seq")
model = AutoModelForSeq2SeqLM.from_pretrained("outputs/dexlit-s2s/seq2seq").cuda().eval()

pure = [
    "أنا رايح البيت دلوقتي عشان تعبان جدا",
    "الحمد لله على كل حال يا صديقي",
    "شلونك اليوم؟ ان شاء الله بخير",
]
out = generate(model, tok, pure, cfg, device="cuda")
ok = sum(a.strip() == b.strip() for a, b in zip(out, pure))
for a, b in zip(pure, out):
    print(("PASS" if a.strip() == b.strip() else "FAIL"), "|", b)
print()
print(f"{ok}/{len(pure)} returned unchanged")

In [ ]:
tests = [
    "كنت انترن في او اي اي اللي هي اورانج انوفيشن ايجيبت و بعدها جالي اوفر",
    "و هنا كانت بداية تعاملي مع السبيتش ايه اي و الارابيك ان ال بي",
    "عندي ميتينج مع المانجر بكرة",
    "طيب خلينا نتقابل بكرة في الكافيه نجهز البريزنتيشن",
    "انك تكتب كود ب سي بلاس بلاس ده مبقاش موجود دلوقتي الناس كلها بتكتب بايثون",
    "ابعتلي على احمد ات جيميل دوت كوم",
    "ادخل على الريبو على جيتهاب و نزلها بجيت بعد ما تعمل انستال للديبنديسيس",
    "انت اتكلمت في توبيك قبل كدة او عملت ايفالويشن قبل كدة لتوبيك زي ده؟",
]
for s, o in zip(tests, generate(model, tok, tests, cfg, device="cuda")):
    print("IN  :", s)
    print("OUT :", o)
    print()

## 9. Evaluate on the held-out test set

In [ ]:
!python scripts/evaluate.py \
    --model-dir outputs/dexlit \
    --data data/processed/test.jsonl \
    --out outputs/dexlit/test_report.json

## 10. Save your work

**Colab wipes the VM when the session ends.** Run at least one of these.

In [ ]:
# Option A -- copy to Google Drive
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p /content/drive/MyDrive/ArabicDeXlit
!cp -r outputs/dexlit-s2s /content/drive/MyDrive/ArabicDeXlit/
print("saved to Drive")

In [ ]:
# Option B -- push the trained model to the Hugging Face Hub
from huggingface_hub import notebook_login, HfApi

notebook_login()

REPO = "mohammedaly22/ArabicDeXlit-base"
api = HfApi()
api.create_repo(REPO, exist_ok=True)
api.upload_folder(folder_path="outputs/dexlit-s2s/seq2seq", repo_id=REPO,
                  commit_message="Add trained ArabicDeXlit model")
print(f"https://huggingface.co/{REPO}")

---

## Troubleshooting

**Out of memory.** Lower `--batch-size` (try 16, then 8) and raise `--grad-accum` to keep the
effective batch size the same. On a T4, `configs/detector_t4.yaml` already freezes the bottom 6
encoder layers, which is the single biggest saving.

**Training is too slow.** Cap the corpus with `--max-train-examples 150000`, or raise
`--freeze-encoder-layers`. Stage 1 dominates the runtime; stage 2 is always quick.

**Pass-through accuracy is low.** The model is editing text it should leave alone. Rebuild with a
higher `--passthrough-ratio` (0.4), or raise `false_edit_penalty` in the config so checkpoint
selection punishes false edits harder.

**The model misses acronyms or emails.** Those categories are rarer in the corpus than plain
code-switching. Generate more with `scripts/synthesize_data.py --categories ACRONYM EMAIL`.

**Colab disconnected mid-run.** Re-run sections 1–3, then 6. Training restarts from scratch, so on
long runs save checkpoints to Drive (section 10) as you go.

---

- **Code:** <https://github.com/MohammedAly22/Arabic-DeXlit>
- **Dataset:** <https://huggingface.co/datasets/mohammedaly22/ArabicDeXlit-Corpus>
- **Model:** <https://huggingface.co/mohammedaly22/ArabicDeXlit-base>